[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/pytorch/lab-p2-tape-microscope.ipynb)

# LAB·P2 · The tape under a microscope

**Hardware:** any machine. Every tensor and every backward pass here runs on plain CPU.

Autograd builds a graph while your Python runs, not before it: every op on a tensor with `requires_grad=True` appends a node, and `backward()` walks that graph from the output back to the leaves. This lab has you read the graph by hand through `grad_fn` and `next_functions`, then deliberately trigger three things the graph refuses to let you get away with, reading each error as autograd's own account of what happened.

Before every reveal cell there is an empty "your prediction" cell above it. Write your answer there, then run the reveal and compare.

In [ ]:
import torch

print(torch.__version__)

## The tape, one node at a time

Every intermediate tensor produced from a `requires_grad` input carries a `grad_fn`: the function that made it, and the node `backward()` will call first. `grad_fn` is not a lone node either; it links to the nodes that fed it through `next_functions`, one entry per input the op took. Walking that list one level down is walking the tape backward by hand, the same walk `backward()` automates.

**your prediction:**

For `x = torch.ones(3, requires_grad=True)` and `y = (x * x).sum()`, name the op you expect `y.grad_fn` to print, and write down what `x.grad` will hold after `y.backward()`. Then, for a three-op program `z = (x * x + 1).sum()`, name the op you expect one level down in `z.grad_fn.next_functions`.

In [ ]:
x = torch.ones(3, requires_grad=True)
y = (x * x).sum()
print(y.grad_fn)     # <SumBackward0 object at ...>: the tape, built as you ran
y.backward()
print(x.grad)        # tensor([2., 2., 2.])

z = (x * x + 1).sum()
print(z.grad_fn)
for fn, _ in z.grad_fn.next_functions:
    print(" ", fn)

`y.grad_fn` names `SumBackward0`: the last op that ran. `x.grad` comes out `[2, 2, 2]` because the derivative of `sum(x * x)` with respect to each entry of `x` is `2x`, and every entry of `x` started at one. Walking `z.grad_fn.next_functions` one level down surfaces the add that fed the sum: the tape is a graph of these links, one per op that ran, not a flat list.

## Three failures, on purpose

The next three exhibits live in the site's museum (`/mistakes/pytorch`). Reproduce each here, writing your prediction before you run it. All three are autograd refusing to walk a graph it no longer trusts, for three different reasons.

**your prediction:**

`x = torch.ones(3, requires_grad=True)`, then `x += 1`: an in-place add directly on a leaf. Will this run? If not, what does the error say?

In [ ]:
x = torch.ones(3, requires_grad=True)
try:
    x += 1   # in-place on the leaf the optimizer would own
except RuntimeError as e:
    print(e)
# RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.

**your prediction:**

`x = torch.ones(3, requires_grad=True)`, `y = x.exp()`, then `y += 1` before `y.sum().backward()`. `exp`'s backward needs the value it saved. What happens?

In [ ]:
x = torch.ones(3, requires_grad=True)
y = x.exp()
try:
    y += 1   # exp's backward needs y; this rewrites it
    y.sum().backward()
except RuntimeError as e:
    print(e)
# RuntimeError: one of the variables needed for gradient computation has been modified by an
# inplace operation: [torch.FloatTensor [3]], which is output 0 of ExpBackward0, is at version 1;
# expected version 0 instead. Hint: enable anomaly detection to find the operation that failed to
# compute its gradient, with torch.autograd.set_detect_anomaly(True).

**your prediction:**

`x = torch.ones(3, requires_grad=True)`, `y = (x * x).sum()`, `y.backward()`, then `y.backward()` again. What happens on the second call?

In [ ]:
x = torch.ones(3, requires_grad=True)
y = (x * x).sum()
y.backward()
try:
    y.backward()   # the tape was freed on the first pass
except RuntimeError as e:
    print(e)
# RuntimeError: Trying to backward through the graph a second time (or directly access saved
# tensors after they have already been freed). Specify retain_graph=True if you need to backward
# through the graph a second time or if you need to access saved tensors after calling backward.

The leaf error guards the parameter itself: a leaf with `requires_grad` is what the optimizer owns, and writing to it in place mid-graph would corrupt a tape that has not run backward yet. The version-counter error guards a saved intermediate: `exp` kept its own output for backward, and the in-place add rewrote it before backward could read it, which the version counter caught. The double-backward error guards the graph as a whole: `backward()` frees each node's saved tensors as it passes them by default, so a second call finds nothing left to walk. All three read as the same policy applied at three grains: leaf, tensor, graph. Fixes follow the same shape too: mutate leaves under `torch.no_grad()` (what `optimizer.step` does, ch 4), replace `y += 1` with `y = y + 1` so the saved value survives, and pass `retain_graph=True` (or recompute the forward) when a second backward is genuinely needed.

## Grads accumulate; zero_grad clears them

`backward()` adds into `.grad`; it does not overwrite it. Call it twice on fresh forward passes without clearing in between and the two gradients sum.

**your prediction:** starting from a fresh `x`, run `y1 = (x * x).sum(); y1.backward()` and read `x.grad`. Then run `y2 = (x * x).sum(); y2.backward()` again without clearing. Predict the second `x.grad`, then predict what it becomes after `x.grad = None` and one more `backward()`.

In [ ]:
x = torch.ones(3, requires_grad=True)
y1 = (x * x).sum()
y1.backward()
print(x.grad)          # tensor([2., 2., 2.])

y2 = (x * x).sum()
y2.backward()
print(x.grad)          # tensor([4., 4., 4.]): accumulated, not overwritten

x.grad = None          # what optimizer.zero_grad() does per parameter
y3 = (x * x).sum()
y3.backward()
print(x.grad)          # tensor([2., 2., 2.]) again

## A custom derivative

`torch.autograd.Function` lets you own a node's backward directly: write `forward` and `backward` as a matched pair, and `gradcheck` compares your analytic gradient against a finite-difference estimate of the same function.

In [ ]:
from torch.autograd import Function
from torch.autograd.gradcheck import gradcheck

class Square(Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * x

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x

x = torch.randn(4, dtype=torch.double, requires_grad=True)
print(gradcheck(Square.apply, (x,)))   # True: the analytic and numeric gradients agree

## Mark it run

Read the two chapters this lab drills: [kernels.rudrite.com/pytorch/autograd](https://kernels.rudrite.com/pytorch/autograd) and [kernels.rudrite.com/pytorch/the-loop](https://kernels.rudrite.com/pytorch/the-loop).